In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")
from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_districtflowname, get_data, get_data_outbkr, get_homebased_tag

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [3]:
def preprocess(df):
    df = get_homebased_tag(df, tag_colname='hb_tag')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='otaz', new_colname='o_district')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='dtaz', new_colname='d_district')
    return df

# survey
data_fullsurvey['Trip'] = preprocess(data_fullsurvey['Trip'])

# bkrcast
data_daysim['Trip'] = preprocess(data_daysim['Trip'])

In [4]:
summary_survey = data_fullsurvey['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

In [5]:
def show_pivot_table(df):
    pivot_survey = (
    df
    .pivot_table(index='o_district', columns='d_district', values='trexpfac', aggfunc='sum', fill_value=0)
    .reindex(index=district_flow_name.values(), columns=district_flow_name.values())
    .fillna(0)
    )

    pivot_survey['Row Total'] = pivot_survey.sum(axis=1)
    pivot_survey.loc['Column Total'] = pivot_survey.sum()
    pivot_survey.columns.name = 'Destination'
    pivot_survey.index.name = 'Origin'

    display(pivot_survey.style.format('{:,.0f}').set_caption('Number of Trips'))
    # show as percent of grand total
    grand_total = pivot_survey.loc['Column Total', 'Row Total']
    if grand_total and grand_total != 0:
        pivot_pct = (pivot_survey / grand_total) * 100
        display(pivot_pct.style.format('{:,.1f}%').set_caption('Trip Share'))

# PSRC Region

## All

In [6]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"389,595","16,820","9,354","92,572","51,196","13,299","130,987","703,823"
Bellevue Downtown,"20,298","32,881","5,253",144,"1,952","1,497","11,586","73,611"
Kirkland,"13,784","7,213","150,522","31,410","12,889","3,835","72,289","291,942"
Redmond,"75,015","10,558","36,001","155,985","15,957","7,576","90,394","391,486"
Seattle (excluding Seattle downtown),"34,249","3,010","13,385","21,630","2,158,314","193,078","332,702","2,756,368"
Seattle downtown,"16,512","2,628","4,325","7,121","187,158","211,394","79,153","508,290"
Rest,"125,351","10,855","68,226","83,048","353,082","71,159","10,106,807","10,818,527"
Column Total,"674,805","83,964","287,065","391,911","2,780,548","501,837","10,823,917","15,544,048"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.5%,0.1%,0.1%,0.6%,0.3%,0.1%,0.8%,4.5%
Bellevue Downtown,0.1%,0.2%,0.0%,0.0%,0.0%,0.0%,0.1%,0.5%
Kirkland,0.1%,0.0%,1.0%,0.2%,0.1%,0.0%,0.5%,1.9%
Redmond,0.5%,0.1%,0.2%,1.0%,0.1%,0.0%,0.6%,2.5%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.1%,13.9%,1.2%,2.1%,17.7%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.2%,1.4%,0.5%,3.3%
Rest,0.8%,0.1%,0.4%,0.5%,2.3%,0.5%,65.0%,69.6%
Column Total,4.3%,0.5%,1.8%,2.5%,17.9%,3.2%,69.6%,100.0%


In [7]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"321,897","48,061","28,715","55,575","35,292","9,645","134,045","633,230"
Bellevue Downtown,"49,700","76,362","11,399","7,889","12,774","2,001","38,274","198,399"
Kirkland,"27,674","10,896","209,125","28,893","18,251","4,261","82,893","381,993"
Redmond,"55,418","8,312","29,477","185,808","12,837","2,546","81,962","376,360"
Seattle (excluding Seattle downtown),"35,289","12,987","18,123","13,497","2,238,290","153,294","357,031","2,828,511"
Seattle downtown,"10,039","2,276","3,916","2,453","154,584","422,565","78,057","673,890"
Rest,"133,182","39,478","81,232","82,195","356,466","79,584","10,271,840","11,043,977"
Column Total,"633,199","198,372","381,987","376,310","2,828,494","673,896","11,044,102","16,136,360"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.0%,0.3%,0.2%,0.3%,0.2%,0.1%,0.8%,3.9%
Bellevue Downtown,0.3%,0.5%,0.1%,0.0%,0.1%,0.0%,0.2%,1.2%
Kirkland,0.2%,0.1%,1.3%,0.2%,0.1%,0.0%,0.5%,2.4%
Redmond,0.3%,0.1%,0.2%,1.2%,0.1%,0.0%,0.5%,2.3%
Seattle (excluding Seattle downtown),0.2%,0.1%,0.1%,0.1%,13.9%,0.9%,2.2%,17.5%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.0%,2.6%,0.5%,4.2%
Rest,0.8%,0.2%,0.5%,0.5%,2.2%,0.5%,63.7%,68.4%
Column Total,3.9%,1.2%,2.4%,2.3%,17.5%,4.2%,68.4%,100.0%


## HBW

In [8]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"22,409","5,061","2,084","3,382","4,666","2,380","23,426","63,407"
Bellevue Downtown,"4,862",499,136,102,170,372,"7,106","13,247"
Kirkland,"2,077",25,768,"11,198","7,663","2,615","15,859","40,205"
Redmond,"3,151",35,"3,600","13,981","6,246","4,416","16,287","47,717"
Seattle (excluding Seattle downtown),"5,286",260,"7,266","7,926","156,124","40,532","77,017","294,413"
Seattle downtown,"1,613",332,"2,615","3,664","33,067","21,905","29,010","92,206"
Rest,"26,981","7,927","16,877","16,381","83,955","28,832","976,836","1,157,789"
Column Total,"66,378","14,140","33,347","56,634","291,892","101,052","1,145,541","1,708,984"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.3%,0.3%,0.1%,0.2%,0.3%,0.1%,1.4%,3.7%
Bellevue Downtown,0.3%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.8%
Kirkland,0.1%,0.0%,0.0%,0.7%,0.4%,0.2%,0.9%,2.4%
Redmond,0.2%,0.0%,0.2%,0.8%,0.4%,0.3%,1.0%,2.8%
Seattle (excluding Seattle downtown),0.3%,0.0%,0.4%,0.5%,9.1%,2.4%,4.5%,17.2%
Seattle downtown,0.1%,0.0%,0.2%,0.2%,1.9%,1.3%,1.7%,5.4%
Rest,1.6%,0.5%,1.0%,1.0%,4.9%,1.7%,57.2%,67.7%
Column Total,3.9%,0.8%,2.0%,3.3%,17.1%,5.9%,67.0%,100.0%


In [9]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"9,681","5,487","2,318","5,529","7,078","3,437","18,736","52,266"
Bellevue Downtown,"4,015","2,550","1,787","2,186","4,178",761,"12,013","27,490"
Kirkland,"2,737","2,633","7,659","3,899","3,698","2,437","10,000","33,063"
Redmond,"4,971","3,032","3,172","9,592","4,757","1,256","18,102","44,882"
Seattle (excluding Seattle downtown),"7,749","6,007","3,515","5,850","166,153","53,358","80,257","322,889"
Seattle downtown,"2,578",818,"1,710",949,"38,005","50,365","28,208","122,633"
Rest,"23,798","17,762","11,327","24,644","92,781","41,324","801,842","1,013,478"
Column Total,"55,529","38,289","31,488","52,649","316,650","152,938","969,158","1,616,701"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.6%,0.3%,0.1%,0.3%,0.4%,0.2%,1.2%,3.2%
Bellevue Downtown,0.2%,0.2%,0.1%,0.1%,0.3%,0.0%,0.7%,1.7%
Kirkland,0.2%,0.2%,0.5%,0.2%,0.2%,0.2%,0.6%,2.0%
Redmond,0.3%,0.2%,0.2%,0.6%,0.3%,0.1%,1.1%,2.8%
Seattle (excluding Seattle downtown),0.5%,0.4%,0.2%,0.4%,10.3%,3.3%,5.0%,20.0%
Seattle downtown,0.2%,0.1%,0.1%,0.1%,2.4%,3.1%,1.7%,7.6%
Rest,1.5%,1.1%,0.7%,1.5%,5.7%,2.6%,49.6%,62.7%
Column Total,3.4%,2.4%,1.9%,3.3%,19.6%,9.5%,59.9%,100.0%


## HBO

In [10]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"144,068","6,808","4,015","20,808","25,331","6,710","57,513","265,254"
Bellevue Downtown,"8,175","5,794","1,630",42,"1,202",791,"2,436","20,069"
Kirkland,"3,822","2,254","64,199","11,877","3,686","1,194","36,327","123,359"
Redmond,"21,255",211,"21,906","76,641","5,483","2,716","37,567","165,778"
Seattle (excluding Seattle downtown),"14,317","1,913","4,452","12,527","1,222,512","74,322","137,449","1,467,492"
Seattle downtown,"12,104",442,"1,621","2,134","75,590","82,012","25,900","199,804"
Rest,"68,565","2,060","30,113","45,778","143,719","24,613","5,077,699","5,392,548"
Column Total,"272,307","19,482","127,937","169,806","1,477,524","192,358","5,374,891","7,634,304"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.9%,0.1%,0.1%,0.3%,0.3%,0.1%,0.8%,3.5%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%
Kirkland,0.1%,0.0%,0.8%,0.2%,0.0%,0.0%,0.5%,1.6%
Redmond,0.3%,0.0%,0.3%,1.0%,0.1%,0.0%,0.5%,2.2%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.2%,16.0%,1.0%,1.8%,19.2%
Seattle downtown,0.2%,0.0%,0.0%,0.0%,1.0%,1.1%,0.3%,2.6%
Rest,0.9%,0.0%,0.4%,0.6%,1.9%,0.3%,66.5%,70.6%
Column Total,3.6%,0.3%,1.7%,2.2%,19.4%,2.5%,70.4%,100.0%


In [11]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"173,247","23,698","16,369","28,673","16,566","3,018","77,871","339,442"
Bellevue Downtown,"25,221","31,525","4,247","2,818","3,204",278,"15,106","82,399"
Kirkland,"15,900","4,064","119,726","13,151","8,826",723,"48,650","211,040"
Redmond,"29,315","2,760","13,150","91,772","2,556",352,"42,312","182,217"
Seattle (excluding Seattle downtown),"15,772","2,458","8,469","2,349","1,255,014","56,009","157,965","1,498,036"
Seattle downtown,"3,214",232,731,325,"63,288","181,174","21,852","270,816"
Rest,"72,500","12,766","44,830","37,141","155,327","18,151","6,024,580","6,365,295"
Column Total,"335,169","77,503","207,522","176,229","1,504,781","259,705","6,388,336","8,949,245"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.9%,0.3%,0.2%,0.3%,0.2%,0.0%,0.9%,3.8%
Bellevue Downtown,0.3%,0.4%,0.0%,0.0%,0.0%,0.0%,0.2%,0.9%
Kirkland,0.2%,0.0%,1.3%,0.1%,0.1%,0.0%,0.5%,2.4%
Redmond,0.3%,0.0%,0.1%,1.0%,0.0%,0.0%,0.5%,2.0%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.0%,14.0%,0.6%,1.8%,16.7%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,0.7%,2.0%,0.2%,3.0%
Rest,0.8%,0.1%,0.5%,0.4%,1.7%,0.2%,67.3%,71.1%
Column Total,3.7%,0.9%,2.3%,2.0%,16.8%,2.9%,71.4%,100.0%


## NHB

In [12]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"193,068","4,628","2,247","68,159","18,869","4,146","34,298","325,415"
Bellevue Downtown,"6,940","26,375","3,487",0,454,334,"2,032","39,622"
Kirkland,"6,878","4,934","74,707","2,692","1,446",26,"19,871","110,555"
Redmond,"47,275","10,311","4,383","59,032","4,228",443,"25,394","151,067"
Seattle (excluding Seattle downtown),"12,426",712,"1,639","1,177","608,642","73,293","108,712","806,601"
Seattle downtown,"2,731","1,853",89,"1,323","74,731","106,994","24,242","211,964"
Rest,"13,735",868,"21,004","8,343","116,861","17,714","3,506,382","3,684,907"
Column Total,"283,053","49,682","107,556","140,727","825,232","202,950","3,720,931","5,330,130"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),3.6%,0.1%,0.0%,1.3%,0.4%,0.1%,0.6%,6.1%
Bellevue Downtown,0.1%,0.5%,0.1%,0.0%,0.0%,0.0%,0.0%,0.7%
Kirkland,0.1%,0.1%,1.4%,0.1%,0.0%,0.0%,0.4%,2.1%
Redmond,0.9%,0.2%,0.1%,1.1%,0.1%,0.0%,0.5%,2.8%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.0%,11.4%,1.4%,2.0%,15.1%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.4%,2.0%,0.5%,4.0%
Rest,0.3%,0.0%,0.4%,0.2%,2.2%,0.3%,65.8%,69.1%
Column Total,5.3%,0.9%,2.0%,2.6%,15.5%,3.8%,69.8%,100.0%


In [13]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"111,655","17,793","9,041","19,245","10,089","3,077","28,837","199,737"
Bellevue Downtown,"19,421","41,769","5,292","2,858","5,035",935,"10,821","86,131"
Kirkland,"8,049","4,123","69,583","10,377","4,618","1,069","19,069","116,888"
Redmond,"18,709","2,484","11,530","78,282","4,717",909,"19,460","136,091"
Seattle (excluding Seattle downtown),"10,259","4,208","5,100","4,574","684,344","39,069","98,543","846,097"
Seattle downtown,"4,142","1,206","1,449","1,156","48,642","185,523","26,154","268,272"
Rest,"26,997","8,585","19,549","18,189","86,062","18,038","2,909,599","3,087,019"
Column Total,"199,232","80,168","121,544","134,681","843,507","248,620","3,112,483","4,740,235"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.4%,0.4%,0.2%,0.4%,0.2%,0.1%,0.6%,4.2%
Bellevue Downtown,0.4%,0.9%,0.1%,0.1%,0.1%,0.0%,0.2%,1.8%
Kirkland,0.2%,0.1%,1.5%,0.2%,0.1%,0.0%,0.4%,2.5%
Redmond,0.4%,0.1%,0.2%,1.7%,0.1%,0.0%,0.4%,2.9%
Seattle (excluding Seattle downtown),0.2%,0.1%,0.1%,0.1%,14.4%,0.8%,2.1%,17.8%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.0%,3.9%,0.6%,5.7%
Rest,0.6%,0.2%,0.4%,0.4%,1.8%,0.4%,61.4%,65.1%
Column Total,4.2%,1.7%,2.6%,2.8%,17.8%,5.2%,65.7%,100.0%


# In-BKR Households

In [14]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)

In [15]:
summary_survey = data_fullsurvey_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [16]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"331,648","15,599","7,771","60,796","28,106","9,333","45,388","498,642"
Bellevue Downtown,"15,438","30,188","5,253",144,772,961,"1,815","54,572"
Kirkland,"9,100","5,412","117,444","29,614","4,300","3,719","24,018","193,607"
Redmond,"44,231","10,558","35,705","129,983","3,922","3,981","45,531","273,912"
Seattle (excluding Seattle downtown),"12,517",768,"4,320","10,170","12,929","17,099","1,333","59,137"
Seattle downtown,"15,607","2,493","3,719","3,560","5,293","6,842","3,275","40,789"
Rest,"47,821",960,"15,766","43,159","1,086","1,653","63,693","174,137"
Column Total,"476,363","65,977","189,979","277,426","56,409","43,589","185,053","1,294,796"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),25.6%,1.2%,0.6%,4.7%,2.2%,0.7%,3.5%,38.5%
Bellevue Downtown,1.2%,2.3%,0.4%,0.0%,0.1%,0.1%,0.1%,4.2%
Kirkland,0.7%,0.4%,9.1%,2.3%,0.3%,0.3%,1.9%,15.0%
Redmond,3.4%,0.8%,2.8%,10.0%,0.3%,0.3%,3.5%,21.2%
Seattle (excluding Seattle downtown),1.0%,0.1%,0.3%,0.8%,1.0%,1.3%,0.1%,4.6%
Seattle downtown,1.2%,0.2%,0.3%,0.3%,0.4%,0.5%,0.3%,3.2%
Rest,3.7%,0.1%,1.2%,3.3%,0.1%,0.1%,4.9%,13.4%
Column Total,36.8%,5.1%,14.7%,21.4%,4.4%,3.4%,14.3%,100.0%


In [17]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"283,919","40,639","25,075","49,375","14,585","6,616","27,961","448,170"
Bellevue Downtown,"41,746","59,682","8,951","7,131","2,179",956,"2,488","123,133"
Kirkland,"24,660","8,833","189,986","25,187","11,273","3,808","26,658","290,405"
Redmond,"48,942","7,573","24,906","155,489","5,391","1,870","15,150","259,321"
Seattle (excluding Seattle downtown),"14,409","2,441","11,123","5,285","6,240",502,"1,177","41,177"
Seattle downtown,"6,709","1,180","3,466","1,706",608,"4,027",319,"18,015"
Rest,"27,807","2,772","26,909","15,138",901,236,"14,073","87,836"
Column Total,"448,192","123,120","290,416","259,311","41,177","18,015","87,826","1,268,057"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),22.4%,3.2%,2.0%,3.9%,1.2%,0.5%,2.2%,35.3%
Bellevue Downtown,3.3%,4.7%,0.7%,0.6%,0.2%,0.1%,0.2%,9.7%
Kirkland,1.9%,0.7%,15.0%,2.0%,0.9%,0.3%,2.1%,22.9%
Redmond,3.9%,0.6%,2.0%,12.3%,0.4%,0.1%,1.2%,20.5%
Seattle (excluding Seattle downtown),1.1%,0.2%,0.9%,0.4%,0.5%,0.0%,0.1%,3.2%
Seattle downtown,0.5%,0.1%,0.3%,0.1%,0.0%,0.3%,0.0%,1.4%
Rest,2.2%,0.2%,2.1%,1.2%,0.1%,0.0%,1.1%,6.9%
Column Total,35.3%,9.7%,22.9%,20.4%,3.2%,1.4%,6.9%,100.0%


## HBW

In [18]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"22,409","5,061","2,084","3,382","1,350","2,352","5,255","41,893"
Bellevue Downtown,"4,862",499,136,102,38,278,146,"6,062"
Kirkland,"2,077",25,768,"11,198","2,184","2,569","3,882","22,703"
Redmond,"3,151",35,"3,600","13,981","1,034","2,159","5,702","29,663"
Seattle (excluding Seattle downtown),"1,228",38,"2,184","1,034",0,0,0,"4,484"
Seattle downtown,"1,586",239,"2,569","1,379",0,0,0,"5,772"
Rest,"4,030",101,"3,882","4,286",0,0,0,"12,299"
Column Total,"39,343","5,998","15,223","35,362","4,606","7,359","14,986","122,877"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),18.2%,4.1%,1.7%,2.8%,1.1%,1.9%,4.3%,34.1%
Bellevue Downtown,4.0%,0.4%,0.1%,0.1%,0.0%,0.2%,0.1%,4.9%
Kirkland,1.7%,0.0%,0.6%,9.1%,1.8%,2.1%,3.2%,18.5%
Redmond,2.6%,0.0%,2.9%,11.4%,0.8%,1.8%,4.6%,24.1%
Seattle (excluding Seattle downtown),1.0%,0.0%,1.8%,0.8%,0.0%,0.0%,0.0%,3.6%
Seattle downtown,1.3%,0.2%,2.1%,1.1%,0.0%,0.0%,0.0%,4.7%
Rest,3.3%,0.1%,3.2%,3.5%,0.0%,0.0%,0.0%,10.0%
Column Total,32.0%,4.9%,12.4%,28.8%,3.7%,6.0%,12.2%,100.0%


In [19]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"9,681","5,487","2,318","5,529","3,493","3,029","5,353","34,890"
Bellevue Downtown,"4,015","2,550","1,787","2,186",446,393,593,"11,970"
Kirkland,"2,737","2,633","7,659","3,899","2,433","2,362","4,660","26,383"
Redmond,"4,971","3,032","3,172","9,592","1,599","1,041","2,974","26,381"
Seattle (excluding Seattle downtown),"2,258",334,"1,618","1,057",0,0,0,"5,267"
Seattle downtown,"1,979",259,"1,583",648,0,0,0,"4,469"
Rest,"3,652",426,"3,208","2,003",0,0,0,"9,289"
Column Total,"29,293","14,721","21,345","24,914","7,971","6,825","13,580","118,649"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),8.2%,4.6%,2.0%,4.7%,2.9%,2.6%,4.5%,29.4%
Bellevue Downtown,3.4%,2.1%,1.5%,1.8%,0.4%,0.3%,0.5%,10.1%
Kirkland,2.3%,2.2%,6.5%,3.3%,2.1%,2.0%,3.9%,22.2%
Redmond,4.2%,2.6%,2.7%,8.1%,1.3%,0.9%,2.5%,22.2%
Seattle (excluding Seattle downtown),1.9%,0.3%,1.4%,0.9%,0.0%,0.0%,0.0%,4.4%
Seattle downtown,1.7%,0.2%,1.3%,0.5%,0.0%,0.0%,0.0%,3.8%
Rest,3.1%,0.4%,2.7%,1.7%,0.0%,0.0%,0.0%,7.8%
Column Total,24.7%,12.4%,18.0%,21.0%,6.7%,5.8%,11.4%,100.0%


## HBO

In [20]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"144,068","6,808","4,015","20,808","20,263","6,595","32,546","235,104"
Bellevue Downtown,"8,175","5,794","1,630",42,473,355,589,"17,057"
Kirkland,"3,822","2,254","64,199","11,877","1,555","1,150","16,611","101,468"
Redmond,"21,255",211,"21,906","76,641","2,137","1,494","13,458","137,103"
Seattle (excluding Seattle downtown),"9,291",389,"2,116","8,632",0,0,0,"20,429"
Seattle downtown,"11,958",401,"1,150",912,0,0,0,"14,420"
Rest,"34,730",691,"8,222","25,364",0,0,115,"69,121"
Column Total,"233,299","16,548","103,238","144,275","24,429","9,593","63,320","594,703"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),24.2%,1.1%,0.7%,3.5%,3.4%,1.1%,5.5%,39.5%
Bellevue Downtown,1.4%,1.0%,0.3%,0.0%,0.1%,0.1%,0.1%,2.9%
Kirkland,0.6%,0.4%,10.8%,2.0%,0.3%,0.2%,2.8%,17.1%
Redmond,3.6%,0.0%,3.7%,12.9%,0.4%,0.3%,2.3%,23.1%
Seattle (excluding Seattle downtown),1.6%,0.1%,0.4%,1.5%,0.0%,0.0%,0.0%,3.4%
Seattle downtown,2.0%,0.1%,0.2%,0.2%,0.0%,0.0%,0.0%,2.4%
Rest,5.8%,0.1%,1.4%,4.3%,0.0%,0.0%,0.0%,11.6%
Column Total,39.2%,2.8%,17.4%,24.3%,4.1%,1.6%,10.6%,100.0%


In [21]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"173,247","23,698","16,369","28,673","6,992","1,946","15,464","266,389"
Bellevue Downtown,"25,221","31,525","4,247","2,818",431,91,428,"64,761"
Kirkland,"15,900","4,064","119,726","13,151","5,932",679,"14,451","173,903"
Redmond,"29,315","2,760","13,150","91,772","1,748",326,"7,850","146,921"
Seattle (excluding Seattle downtown),"6,603",382,"5,700","1,579",0,0,0,"14,264"
Seattle downtown,"2,094",87,682,301,0,0,0,"3,164"
Rest,"15,194",387,"14,311","7,387",0,0,0,"37,279"
Column Total,"267,574","62,903","174,185","145,681","15,103","3,042","38,193","706,681"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),24.5%,3.4%,2.3%,4.1%,1.0%,0.3%,2.2%,37.7%
Bellevue Downtown,3.6%,4.5%,0.6%,0.4%,0.1%,0.0%,0.1%,9.2%
Kirkland,2.2%,0.6%,16.9%,1.9%,0.8%,0.1%,2.0%,24.6%
Redmond,4.1%,0.4%,1.9%,13.0%,0.2%,0.0%,1.1%,20.8%
Seattle (excluding Seattle downtown),0.9%,0.1%,0.8%,0.2%,0.0%,0.0%,0.0%,2.0%
Seattle downtown,0.3%,0.0%,0.1%,0.0%,0.0%,0.0%,0.0%,0.4%
Rest,2.2%,0.1%,2.0%,1.0%,0.0%,0.0%,0.0%,5.3%
Column Total,37.9%,8.9%,24.6%,20.6%,2.1%,0.4%,5.4%,100.0%


## NHB

In [22]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"135,121","3,408",664,"36,383","5,423",323,"6,819","188,141"
Bellevue Downtown,"2,080","23,682","3,487",0,136,329,"1,067","30,780"
Kirkland,"2,193","3,133","41,630",896,561,0,"3,293","51,707"
Redmond,"16,491","10,311","4,088","33,030",751,328,"15,224","80,223"
Seattle (excluding Seattle downtown),"1,038",215,20,504,"12,929","17,099","1,333","33,139"
Seattle downtown,"2,000","1,853",0,"1,269","5,293","6,842","3,275","20,533"
Rest,"8,295",169,"3,430","1,079","1,086","1,653","63,578","79,290"
Column Total,"167,218","42,771","53,319","73,161","26,179","26,574","94,591","483,812"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),27.9%,0.7%,0.1%,7.5%,1.1%,0.1%,1.4%,38.9%
Bellevue Downtown,0.4%,4.9%,0.7%,0.0%,0.0%,0.1%,0.2%,6.4%
Kirkland,0.5%,0.6%,8.6%,0.2%,0.1%,0.0%,0.7%,10.7%
Redmond,3.4%,2.1%,0.8%,6.8%,0.2%,0.1%,3.1%,16.6%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.1%,2.7%,3.5%,0.3%,6.8%
Seattle downtown,0.4%,0.4%,0.0%,0.3%,1.1%,1.4%,0.7%,4.2%
Rest,1.7%,0.0%,0.7%,0.2%,0.2%,0.3%,13.1%,16.4%
Column Total,34.6%,8.8%,11.0%,15.1%,5.4%,5.5%,19.6%,100.0%


In [23]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"73,677","10,371","5,401","13,045","3,005","1,547","5,934","112,980"
Bellevue Downtown,"11,467","25,089","2,844","2,100",964,450,"1,310","44,224"
Kirkland,"5,035","2,060","50,444","6,671","1,970",740,"5,691","72,611"
Redmond,"12,233","1,745","6,959","47,963","1,288",476,"3,492","74,156"
Seattle (excluding Seattle downtown),"4,612","1,441","2,984","1,986","6,240",502,"1,177","18,942"
Seattle downtown,"2,552",817,"1,181",736,608,"4,027",319,"10,240"
Rest,"7,961","1,824","7,864","5,090",901,236,"14,073","37,949"
Column Total,"117,537","43,347","77,677","77,591","14,976","7,978","31,996","371,102"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),19.9%,2.8%,1.5%,3.5%,0.8%,0.4%,1.6%,30.4%
Bellevue Downtown,3.1%,6.8%,0.8%,0.6%,0.3%,0.1%,0.4%,11.9%
Kirkland,1.4%,0.6%,13.6%,1.8%,0.5%,0.2%,1.5%,19.6%
Redmond,3.3%,0.5%,1.9%,12.9%,0.3%,0.1%,0.9%,20.0%
Seattle (excluding Seattle downtown),1.2%,0.4%,0.8%,0.5%,1.7%,0.1%,0.3%,5.1%
Seattle downtown,0.7%,0.2%,0.3%,0.2%,0.2%,1.1%,0.1%,2.8%
Rest,2.1%,0.5%,2.1%,1.4%,0.2%,0.1%,3.8%,10.2%
Column Total,31.7%,11.7%,20.9%,20.9%,4.0%,2.1%,8.6%,100.0%


# Outside-BKR Households

In [24]:
data_daysim_outbkr, data_survey_outbkr, data_fullsurvey_outbkr = \
    get_data_outbkr(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, taz_subarea=taz_subarea)

In [25]:
summary_survey = data_fullsurvey_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [26]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"57,947","1,221","1,583","31,776","23,090","3,965","85,600","205,181"
Bellevue Downtown,"4,860","2,693",0,0,"1,180",536,"9,770","19,039"
Kirkland,"4,685","1,801","33,077","1,796","8,589",116,"48,271","98,335"
Redmond,"30,784",0,295,"26,002","12,034","3,595","44,863","117,574"
Seattle (excluding Seattle downtown),"21,732","2,242","9,065","11,460","2,145,385","175,979","331,368","2,697,231"
Seattle downtown,905,135,606,"3,561","181,865","204,552","75,878","467,501"
Rest,"77,530","9,895","52,460","39,889","351,996","69,506","10,043,114","10,644,390"
Column Total,"198,443","17,987","97,087","114,484","2,724,139","458,248","10,638,864","14,249,252"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.4%,0.0%,0.0%,0.2%,0.2%,0.0%,0.6%,1.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.1%,0.1%
Kirkland,0.0%,0.0%,0.2%,0.0%,0.1%,0.0%,0.3%,0.7%
Redmond,0.2%,0.0%,0.0%,0.2%,0.1%,0.0%,0.3%,0.8%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.1%,15.1%,1.2%,2.3%,18.9%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.3%,1.4%,0.5%,3.3%
Rest,0.5%,0.1%,0.4%,0.3%,2.5%,0.5%,70.5%,74.7%
Column Total,1.4%,0.1%,0.7%,0.8%,19.1%,3.2%,74.7%,100.0%


In [27]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"37,978","7,422","3,640","6,200","20,707","3,029","106,084","185,060"
Bellevue Downtown,"7,954","16,680","2,448",758,"10,595","1,045","35,786","75,266"
Kirkland,"3,014","2,063","19,139","3,706","6,978",453,"56,235","91,588"
Redmond,"6,476",739,"4,571","30,319","7,446",676,"66,812","117,039"
Seattle (excluding Seattle downtown),"20,880","10,546","7,000","8,212","2,232,050","152,792","355,854","2,787,334"
Seattle downtown,"3,330","1,096",450,747,"153,976","418,538","77,738","655,875"
Rest,"105,375","36,706","54,323","67,057","355,565","79,348","10,257,767","10,956,141"
Column Total,"185,007","75,252","91,571","116,999","2,787,317","655,881","10,956,276","14,868,303"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.3%,0.0%,0.0%,0.0%,0.1%,0.0%,0.7%,1.2%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.1%,0.0%,0.2%,0.5%
Kirkland,0.0%,0.0%,0.1%,0.0%,0.0%,0.0%,0.4%,0.6%
Redmond,0.0%,0.0%,0.0%,0.2%,0.1%,0.0%,0.4%,0.8%
Seattle (excluding Seattle downtown),0.1%,0.1%,0.0%,0.1%,15.0%,1.0%,2.4%,18.7%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.0%,2.8%,0.5%,4.4%
Rest,0.7%,0.2%,0.4%,0.5%,2.4%,0.5%,69.0%,73.7%
Column Total,1.2%,0.5%,0.6%,0.8%,18.7%,4.4%,73.7%,100.0%


## HBW

In [28]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"3,316",28,"18,170","21,514"
Bellevue Downtown,0,0,0,0,133,93,"6,960","7,186"
Kirkland,0,0,0,0,"5,479",46,"11,977","17,502"
Redmond,0,0,0,0,"5,213","2,257","10,585","18,055"
Seattle (excluding Seattle downtown),"4,058",222,"5,082","6,892","156,124","40,532","77,017","289,929"
Seattle downtown,28,93,46,"2,285","33,067","21,905","29,010","86,433"
Rest,"22,950","7,826","12,995","12,095","83,955","28,832","976,836","1,145,490"
Column Total,"27,035","8,142","18,123","21,272","287,286","93,693","1,130,556","1,586,107"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,1.1%,1.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.5%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.3%,0.0%,0.8%,1.1%
Redmond,0.0%,0.0%,0.0%,0.0%,0.3%,0.1%,0.7%,1.1%
Seattle (excluding Seattle downtown),0.3%,0.0%,0.3%,0.4%,9.8%,2.6%,4.9%,18.3%
Seattle downtown,0.0%,0.0%,0.0%,0.1%,2.1%,1.4%,1.8%,5.4%
Rest,1.4%,0.5%,0.8%,0.8%,5.3%,1.8%,61.6%,72.2%
Column Total,1.7%,0.5%,1.1%,1.3%,18.1%,5.9%,71.3%,100.0%


In [29]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"3,585",408,"13,383","17,376"
Bellevue Downtown,0,0,0,0,"3,732",368,"11,420","15,520"
Kirkland,0,0,0,0,"1,265",75,"5,340","6,680"
Redmond,0,0,0,0,"3,158",215,"15,128","18,501"
Seattle (excluding Seattle downtown),"5,491","5,673","1,897","4,793","166,153","53,358","80,257","317,622"
Seattle downtown,599,559,127,301,"38,005","50,365","28,208","118,164"
Rest,"20,146","17,336","8,119","22,641","92,781","41,324","801,842","1,004,189"
Column Total,"26,236","23,568","10,143","27,735","308,679","146,113","955,578","1,498,052"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,0.9%,1.2%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,0.8%,1.0%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.4%,0.4%
Redmond,0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,1.0%,1.2%
Seattle (excluding Seattle downtown),0.4%,0.4%,0.1%,0.3%,11.1%,3.6%,5.4%,21.2%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,2.5%,3.4%,1.9%,7.9%
Rest,1.3%,1.2%,0.5%,1.5%,6.2%,2.8%,53.5%,67.0%
Column Total,1.8%,1.6%,0.7%,1.9%,20.6%,9.8%,63.8%,100.0%


## HBO

In [30]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"5,068",115,"24,967","30,150"
Bellevue Downtown,0,0,0,0,729,437,"1,846","3,012"
Kirkland,0,0,0,0,"2,131",44,"19,716","21,891"
Redmond,0,0,0,0,"3,345","1,222","24,108","28,676"
Seattle (excluding Seattle downtown),"5,026","1,523","2,336","3,894","1,222,512","74,322","137,449","1,447,062"
Seattle downtown,147,42,471,"1,222","75,590","82,012","25,900","185,384"
Rest,"33,835","1,369","21,891","20,414","143,719","24,613","5,077,584","5,323,426"
Column Total,"39,008","2,934","24,699","25,531","1,453,095","182,765","5,311,571","7,039,602"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.4%,0.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%,0.3%
Redmond,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%,0.4%
Seattle (excluding Seattle downtown),0.1%,0.0%,0.0%,0.1%,17.4%,1.1%,2.0%,20.6%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.1%,1.2%,0.4%,2.6%
Rest,0.5%,0.0%,0.3%,0.3%,2.0%,0.3%,72.1%,75.6%
Column Total,0.6%,0.0%,0.4%,0.4%,20.6%,2.6%,75.5%,100.0%


In [31]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"9,574","1,072","62,407","73,053"
Bellevue Downtown,0,0,0,0,"2,773",187,"14,678","17,638"
Kirkland,0,0,0,0,"2,894",44,"34,199","37,137"
Redmond,0,0,0,0,808,26,"34,462","35,296"
Seattle (excluding Seattle downtown),"9,169","2,076","2,769",770,"1,255,014","56,009","157,965","1,483,772"
Seattle downtown,"1,120",145,49,24,"63,288","181,174","21,852","267,652"
Rest,"57,306","12,379","30,519","29,754","155,327","18,151","6,024,580","6,328,016"
Column Total,"67,595","14,600","33,337","30,548","1,489,678","256,663","6,350,143","8,242,564"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.8%,0.9%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.2%,0.2%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.5%
Redmond,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.4%
Seattle (excluding Seattle downtown),0.1%,0.0%,0.0%,0.0%,15.2%,0.7%,1.9%,18.0%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,0.8%,2.2%,0.3%,3.2%
Rest,0.7%,0.2%,0.4%,0.4%,1.9%,0.2%,73.1%,76.8%
Column Total,0.8%,0.2%,0.4%,0.4%,18.1%,3.1%,77.0%,100.0%


## NHB

In [32]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"57,947","1,221","1,583","31,776","13,445","3,823","27,479","137,274"
Bellevue Downtown,"4,860","2,693",0,0,318,5,965,"8,842"
Kirkland,"4,685","1,801","33,077","1,796",885,26,"16,577","58,848"
Redmond,"30,784",0,295,"26,002","3,476",115,"10,170","70,844"
Seattle (excluding Seattle downtown),"11,388",497,"1,619",673,"595,713","56,193","107,379","773,461"
Seattle downtown,731,0,89,54,"69,438","100,152","20,967","191,431"
Rest,"5,440",700,"17,574","7,264","115,776","16,061","3,442,803","3,605,617"
Column Total,"115,835","6,911","54,237","67,566","799,052","176,376","3,626,340","4,846,317"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.2%,0.0%,0.0%,0.7%,0.3%,0.1%,0.6%,2.8%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.2%
Kirkland,0.1%,0.0%,0.7%,0.0%,0.0%,0.0%,0.3%,1.2%
Redmond,0.6%,0.0%,0.0%,0.5%,0.1%,0.0%,0.2%,1.5%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.0%,12.3%,1.2%,2.2%,16.0%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.4%,2.1%,0.4%,4.0%
Rest,0.1%,0.0%,0.4%,0.1%,2.4%,0.3%,71.0%,74.4%
Column Total,2.4%,0.1%,1.1%,1.4%,16.5%,3.6%,74.8%,100.0%


In [33]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"37,978","7,422","3,640","6,200","7,084","1,530","22,903","86,757"
Bellevue Downtown,"7,954","16,680","2,448",758,"4,071",485,"9,511","41,907"
Kirkland,"3,014","2,063","19,139","3,706","2,648",329,"13,378","44,277"
Redmond,"6,476",739,"4,571","30,319","3,429",433,"15,968","61,935"
Seattle (excluding Seattle downtown),"5,647","2,767","2,116","2,588","678,104","38,567","97,366","827,155"
Seattle downtown,"1,590",389,268,420,"48,034","181,496","25,835","258,032"
Rest,"19,036","6,761","11,685","13,099","85,161","17,802","2,895,526","3,049,070"
Column Total,"81,695","36,821","43,867","57,090","828,531","240,642","3,080,487","4,369,133"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.9%,0.2%,0.1%,0.1%,0.2%,0.0%,0.5%,2.0%
Bellevue Downtown,0.2%,0.4%,0.1%,0.0%,0.1%,0.0%,0.2%,1.0%
Kirkland,0.1%,0.0%,0.4%,0.1%,0.1%,0.0%,0.3%,1.0%
Redmond,0.1%,0.0%,0.1%,0.7%,0.1%,0.0%,0.4%,1.4%
Seattle (excluding Seattle downtown),0.1%,0.1%,0.0%,0.1%,15.5%,0.9%,2.2%,18.9%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.1%,4.2%,0.6%,5.9%
Rest,0.4%,0.2%,0.3%,0.3%,1.9%,0.4%,66.3%,69.8%
Column Total,1.9%,0.8%,1.0%,1.3%,19.0%,5.5%,70.5%,100.0%
